# Unbinned Pointing Cut Example

This example reads DC4 unbinned FITS files, builds good time intervals for a source using a 60 degree field-of-view cut, and writes the selected events to unbinned FITS files.

The FOV selection uses `GoodTimeInterval.from_pointing_cut`, with the default example value `max_offaxis = 60 * u.deg`, to keep events when the source is inside the FOV.

**When making a pointing cut in an analysis (e.g., spectral analysis), the data, background, and orientation files need to all be cut self-consistently.**

In [ ]:
from pathlib import Path
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table

from cosipy import UnBinnedData
from cosipy.event_selection import GoodTimeInterval
from cosipy.spacecraftfile import SpacecraftHistory


In [ ]:
### Input files to apply pointing cuts
data_file = Path(
    "dc4_mock_dataset_3months_unbinned_data_filtered_with_SAAcut_time_ordered.fits.gz"
)

bkg_file = Path(
    "Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz"
)

orientation_file = Path(
    "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
)

### Output prefixes for the unbinned files after the pointing cut.
### The format and extension are set by `unbinned_output` in inputs.yaml.
unbinned_output_prefix = Path(
    "dc4_mock_dataset_3months_unbinned_data_filtered_with_SAAcut_time_ordered_NGC4151_cut"
)
unbinned_bkg_output_prefix = Path(
    "Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck_NGC4151_cut"
)

### This config is only used to initialize the DataIO object
config_file = Path("inputs.yaml")

Change details below for the source to be analyzed

In [ ]:
source_name = "NGC 4151"
source_coord = SkyCoord(l=155.07 * u.deg, b=75.06 * u.deg, frame="galactic")

### Default FOV cut used by the pointing-cut helper.
max_offaxis = 60 * u.deg
earth_occ = True

Read the unbinned FITS data and open the spacecraft orientation over the event time range.

In [ ]:
analysis = UnBinnedData(str(config_file))

analysis.cosi_dataset = analysis.get_dict(str(data_file))
time_tags = analysis.cosi_dataset["TimeTags"]

Load in the orientation file.


In [ ]:
orientation = SpacecraftHistory.open(orientation_file)

print(f"Time-cut orientation pointings: {len(orientation.obstime):,}")

Build the GTIs where the source is inside the FOV.

In [ ]:
source_gti = GoodTimeInterval.from_pointing_cut(
    source_coord,
    orientation,
    max_offaxis,
    earth_occ=earth_occ,
)

pointing_cut_orientation = orientation.apply_gti(source_gti)
gti_livetime = pointing_cut_orientation.cumulative_livetime().to_value(u.s)
total_livetime = orientation.cumulative_livetime().to_value(u.s)
livetime_fraction = gti_livetime / total_livetime

print(f"Source: {source_name}")
print(f"FOV cut: off-axis <= {max_offaxis.to_value(u.deg):.1f} deg")
print(f"GTI intervals: {len(source_gti)}")
print(f"GTI livetime: {gti_livetime:,.1f} s")
print(f"Total livetime: {total_livetime:,.1f} s")
print(f"Livetime fraction in FOV: {livetime_fraction:.4f}")

Apply the GTI to the mock-data file. This uses the same half-open convention as `select_data_time`: events are kept for `start <= TimeTags < stop`.


In [ ]:
in_fov_mask = source_gti.contains(analysis.cosi_dataset["TimeTags"])
n_in_fov = np.count_nonzero(in_fov_mask)
n_out_fov = len(in_fov_mask) - n_in_fov

analysis.cosi_dataset = {
    key: values[in_fov_mask]
    for key, values in analysis.cosi_dataset.items()
}
selected_time_tags = analysis.cosi_dataset["TimeTags"]
analysis.tmin = float(np.min(selected_time_tags))
analysis.tmax = float(np.max(selected_time_tags))

print(f"Events in FOV: {n_in_fov:,}")
print(f"Events outside FOV: {n_out_fov:,}")

In [ ]:
analysis.write_unbinned_output(unbinned_output_prefix)

print(f"Wrote: {unbinned_output_prefix}.fits.gz")

del analysis.cosi_dataset, in_fov_mask

Apply the same pointing-cut time intervals to the background file and save the selected events without binning.


In [ ]:
background = UnBinnedData(str(config_file))

background.cosi_dataset = background.get_dict(str(bkg_file))
bkg_in_fov_mask = source_gti.contains(background.cosi_dataset["TimeTags"])

background.cosi_dataset = {
    key: values[bkg_in_fov_mask]
    for key, values in background.cosi_dataset.items()
}
selected_bkg_time_tags = background.cosi_dataset["TimeTags"]
background.tmin = float(np.min(selected_bkg_time_tags))
background.tmax = float(np.max(selected_bkg_time_tags))

background.write_unbinned_output(unbinned_bkg_output_prefix)

print(f"Background events in FOV: {np.count_nonzero(bkg_in_fov_mask):,}")
print(f"Background events outside FOV: {len(bkg_in_fov_mask) - np.count_nonzero(bkg_in_fov_mask):,}")
print(f"Wrote: {unbinned_bkg_output_prefix}.fits.gz")

del background.cosi_dataset, bkg_in_fov_mask

Load the saved pointing-cut unbinned FITS files and verify their event counts.


In [ ]:
mock_events = Table.read(f"{unbinned_output_prefix}.fits.gz")
background_events = Table.read(f"{unbinned_bkg_output_prefix}.fits.gz")

print(f"Saved mock-data events: {len(mock_events):,}")
print(f"Saved background events: {len(background_events):,}")

Save the pointing cut orientation file if needed. The time cut can be performed and used on the orientation file in analysis directly as shown above.

In [ ]:
orientation_output_file = "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA_NGC4151_cut"
pointing_cut_orientation.write_fits(orientation_output_file, overwrite=True)
print(f"Wrote: {orientation_output_file}")